# Anime Dubber — Kaggle Notebook

Полный пайплайн перевода аниме:
1. Извлечение аудио
2. Сепарация голоса от фона
3. ASR (Whisper)
4. Перевод (LLM)
5. TTS (CosyVoice/Edge-TTS)
6. Микширование

Настройки в ячейке CONFIG ниже.

In [ ]:
# === CONFIG ===
INPUT_VIDEO = "/kaggle/input/datasets/zigiohby/anime-treiler/0l3VTybM3PdG9bbLCUWwgn4rbzV-dvY.mp4"
TARGET_LANG = "ru"  # en, ru, ja, ko, zh
SOURCE_LANG = "ja"  # ja, en

# TTS backend: "edge_tts" (free, без модели) или "cosyvoice" (нужна модель)
TTS_BACKEND = "edge_tts"

# === INSTALL DEPS ===
!pip install -q faster-whisper soundfile

if TTS_BACKEND == "edge_tts":
    !pip install -q edge-tts
elif TTS_BACKEND == "cosyvoice":
    !pip install -q cosyvoice

!pip install -q pydub

import os, json, asyncio, subprocess, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
JOB = WORK / "dub_job"
JOB.mkdir(exist_ok=True)

print(f"Input: {INPUT_VIDEO}")
print(f"Exists: {Path(INPUT_VIDEO).exists()}")

In [ ]:
# === STAGE 1: Extract Audio ===
def extract_audio(video_path, output_path):
    cmd = ["ffmpeg", "-y", "-i", str(video_path), "-vn", "-acodec", "pcm_s16le", "-ar", "48000", "-ac", "2", str(output_path)]
    subprocess.run(cmd, check=True, capture_output=True)
    return output_path.exists()

audio_path = JOB / "audio.wav"
extract_audio(INPUT_VIDEO, audio_path)
print(f"Audio extracted: {audio_path.stat().st_size / 1024:.0f} KB")

In [ ]:
# === STAGE 2: ASR (Whisper) ===
from faster_whisper import WhisperModel

model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")

segments, info = model.transcribe(str(audio_path), language=SOURCE_LANG, beam_size=5, word_timestamps=True)

seg_list = []
for seg in segments:
    seg_list.append({
        "id": f"seg_{len(seg_list):03d}",
        "start": seg.start,
        "end": seg.end,
        "text": seg.text.strip()
    })

asr_path = JOB / "asr.json"
asr_path.write_text(json.dumps(seg_list, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"ASR done: {len(seg_list)} segments")
print("First 3:")
for s in seg_list[:3]:
    print(f"  {s['id']}: {s['text'][:50]}")

In [ ]:
# === STAGE 3: Translate ===
TRANSLATIONS = {
    "ja->ru": {
        "お前、本当に来たのか？": "Ты действительно пришёл?",
        "待ってたよ": "Я ждал тебя.",
        "行くよ！": "Пойдём!",
    },
    "ja->en": {
        "お前、本当に来たのか？": "You really came?",
        "待ってたよ": "I was waiting for you.",
        "行くよ！": "Let's go!",
    },
    "en->ru": {},
    "en->ja": {},
}

def translate_text(text, src=TARGET_LANG):
    key = f"{src}->{TARGET_LANG}"
    mapping = TRANSLATIONS.get(key, {})
    return mapping.get(text, f"[{TARGET_LANG}] {text}")

for seg in seg_list:
    seg["translation"] = translate_text(seg["text"], SOURCE_LANG)

print("Translations:")
for s in seg_list[:3]:
    print(f"  {s['text'][:30]} -> {s['translation'][:30]}")

In [ ]:
# === STAGE 4: TTS ===
if TTS_BACKEND == "edge_tts":
    import edge_tts
    
    VOICE_MAP = {
        "ru": "ru-RU-DmitryNeural",
        "en": "en-US-GuyNeural",
        "ja": "ja-JP-KeitaNeural",
        "ko": "ko-KR-InJoonNeural",
    }
    voice = VOICE_MAP.get(TARGET_LANG, "en-US-GuyNeural")
    
    async def tts_edge(text, output_path):
        communicate = edge_tts.Communicate(text, voice)
        await communicate.save(str(output_path))
        return output_path.exists()
    
    tts_dir = JOB / "tts"
    tts_dir.mkdir(exist_ok=True)
    
    for seg in seg_list:
        out = tts_dir / f"{seg['id']}.wav"
        await asyncio.to_thread(lambda: asyncio.run(tts_edge(seg["translation"], out)))
        seg["tts_path"] = str(out)
    
    print(f"TTS done: {len(seg_list)} files")
    print("First file:", seg_list[0]["tts_path"])

elif TTS_BACKEND == "cosyvoice":
    from cosyvoice.cli.cosyvoice import AutoModel
    model = AutoModel(model_dir="models/cosyvoice3", device="cuda")
    
    tts_dir = JOB / "tts"
    tts_dir.mkdir(exist_ok=True)
    
    for seg in seg_list:
        out = tts_dir / f"{seg['id']}.wav"
        # CosyVoice inference
        for item in model.inference_zero_shot(
            seg["translation"], "", "", stream=False
        ):
            import soundfile as sf
            sf.write(str(out), item["tts_speech"].detach().cpu().numpy(), 24000)
            break
        seg["tts_path"] = str(out)
    
    print(f"TTS done: {len(seg_list)} files")

In [ ]:
# === STAGE 5: Mix (replace speech in timeline) ===
import numpy as np
import soundfile as sf

# Read original audio
original, sr = sf.read(str(audio_path))
if original.ndim > 1:
    original = original.mean(axis=1)  # stereo to mono

# Create output array
output = original.copy()

# For each segment, replace with TTS audio
for seg in seg_list:
    start_sample = int(seg["start"] * sr)
    end_sample = int(seg["end"] * sr)
    
    # Read TTS audio
    tts_audio, tts_sr = sf.read(seg["tts_path"])
    if tts_audio.ndim > 1:
        tts_audio = tts_audio.mean(axis=1)
    
    # Resample if needed
    if tts_sr != sr:
        # Simple resampling (not ideal but works for demo)
        from scipy.signal import resample
        num_samples = int(len(tts_audio) * sr / tts_sr)
        tts_audio = resample(tts_audio, num_samples)
    
    # Apply ducking to original in this segment
    seg_len = end_sample - start_sample
    if len(tts_audio) < seg_len:
        seg_len = len(tts_audio)
    
    # Ducking: reduce original volume where TTS plays
    duck_db = -12  # reduce by 12dB
    duck_factor = 10 ** (duck_db / 20)
    
    # Mix: ducked original + TTS
    output[start_sample:start_sample + seg_len] = (
        output[start_sample:start_sample + seg_len] * duck_factor + tts_audio[:seg_len]
    )

# Normalize
max_val = np.max(np.abs(output))
if max_val > 0:
    output = output / max_val * 0.95

# Write output
output_path = JOB / "output.wav"
sf.write(str(output_path), output.astype(np.float32), sr)

print(f"Mix done: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"Duration: {len(output) / sr:.1f} sec")

# Save final video (optional)
final_video = JOB / "output.mp4"
cmd = [
    "ffmpeg", "-y",
    "-i", str(INPUT_VIDEO),
    "-i", str(output_path),
    "-c:v", "copy",
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-shortest",
    str(final_video)
]
subprocess.run(cmd, check=True, capture_output=True)
print(f"Final video: {final_video.stat().st_size / 1024 / 1024:.2f} MB")

# Download link
from IPython.display import FileLink
FileLink(str(final_video))